# Pipeline for OpenTargets Data

In [2]:
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ID = 'gene-expression-big-data'
client = bigquery.Client(project=PROJECT_ID)

# Pull the table
cpm_matrix = client.query("""
    SELECT *
    FROM `gene-expression-big-data.raw_gene_data.cpm_matrix_table`
""").to_dataframe()

print(cpm_matrix.shape)
cpm_matrix.head()

(78986, 99)


,gene_id,gene_name,gene_biotype,ZDS2_1_cpm,ZDS2_1_count,SP1R_1_cpm,SP1R_1_count,SP1R_2_cpm,SP1R_2_count,nZF36_1_cpm,...,hATF555R_1_cpm,hATF555R_1_count,hATF555R_2_cpm,hATF555R_2_count,hATF555Q_1_cpm,hATF555Q_1_count,hATF555Q_2_cpm,hATF555Q_2_count,ZDS2_2_cpm,ZDS2_2_count
0,ENSG00000000005,TNMD,protein_coding,0.104181,1.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0
1,ENSG00000000938,FGR,protein_coding,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.094811,1.0,0.000000,0.0,0.0,0.0,0.00000,0.0,0.000000,0.0
2,ENSG00000001626,CFTR,protein_coding,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.187719,2.0,0.0,0.0,0.00000,0.0,0.000000,0.0
3,ENSG00000002745,WNT16,protein_coding,0.000000,0.0,0.0,0.0,0.0,0.0,0.000000,...,0.000000,0.0,0.000000,0.0,0.0,0.0,0.10054,1.0,0.000000,0.0
4,ENSG00000002746,HECW1,protein_coding,0.000000,0.0,0.0,0.0,0.0,0.0,0.098459,...,0.000000,0.0,0.093859,1.0,0.0,0.0,0.00000,0.0,0.111532,1.0


In [3]:
cpm_matrix = cpm_matrix.loc[:, ~cpm_matrix.columns.str.endswith('_count')]
print(cpm_matrix.shape)

(78986, 51)


In [80]:
# DONT NEED TO RUN THIS CODE SINCE IT IS SAVED TO disease_associations.parquet


gene_list = cpm_matrix['gene_id'].tolist()

ot_query = """
SELECT
  a.targetId AS gene_id,
  dis.name AS disease_name,
  a.associationScore AS association_score
FROM `open-targets-prod.platform.association_overall_direct` AS a
JOIN `open-targets-prod.platform.disease` AS dis
  ON a.diseaseId = dis.id
WHERE a.targetId IN UNNEST(@gene_list)
AND a.associationScore > 0.1
ORDER BY gene_id, association_score DESC
"""

job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter("gene_list", "STRING", gene_list)]
)

associations = client.query(ot_query, job_config=job_config).to_dataframe()
associations.to_parquet('disease_associations.parquet')  # cache it
print(f"{len(associations)} associations across {associations['gene_id'].nunique()} genes")

705267 associations across 18846 genes


In [81]:
disease_associations = pd.read_parquet("disease_associations.parquet")
print(disease_associations.shape)
disease_associations.head()

(705267, 3)


,gene_id,disease_name,association_score
0,ENSG00000000003,hypertension,0.431492
1,ENSG00000000003,low density lipoprotein cholesterol measurement,0.431377
2,ENSG00000000003,glomerular filtration rate,0.429455
3,ENSG00000000003,serum creatinine amount,0.345194
4,ENSG00000000003,body height,0.345194


In [82]:
cpm = pd.read_parquet('sample_by_gene.parquet')

In [83]:
# Drop ERCC spike-ins
gene_cols = [c for c in cpm.columns if c.startswith('ENSG')]
print(f"Real genes: {len(gene_cols)}")

# Confirm sample group composition
print(cpm['sample_type'].value_counts())

# Separate control (SP1R replicates) and treatment (everything else)
control_mask = cpm['sample_type'] == 'SP1R'
control_means = cpm.loc[control_mask, gene_cols].mean(axis=0)
treatment_means = cpm.loc[~control_mask, gene_cols].mean(axis=0)

# Build per-gene scored table
genes = pd.DataFrame({
    'gene_id': gene_cols,
    'control_mean': control_means.values,
    'treatment_mean': treatment_means.values,
})

# Suppression (efficacy) and deviation (safety cost), safe against divide-by-zero
genes['suppression'] = np.where(
    genes['control_mean'] > 0,
    ((genes['control_mean'] - genes['treatment_mean']) / genes['control_mean']).clip(lower=0),
    0
)
genes['deviation'] = np.where(
    genes['control_mean'] > 0,
    np.abs(genes['treatment_mean'] - genes['control_mean']) / genes['control_mean'],
    0
)

print(genes.shape)
genes.head()

Real genes: 78893
sample_type
nZFD96      6
Base        2
SP1R        2
ZDS2        2
hATF555Q    2
Control     2
hATF555R    2
hATF561     2
nZF105      2
hATF567     2
nZF145      2
nZF147      2
nZF148      2
nZF139      2
nZF151      2
nZF153      2
nZF156      2
nZF154      2
nZF36       2
nZF42       2
nZF81       2
nZF93       2
Name: count, dtype: int64
(78893, 5)


,gene_id,control_mean,treatment_mean,suppression,deviation
0,ENSG00000000003,6.880125,8.381852,0.000000,0.218270
1,ENSG00000000005,0.000000,0.010856,0.000000,0.000000
2,ENSG00000000419,52.309032,56.699323,0.000000,0.083930
3,ENSG00000000457,5.151389,5.353453,0.000000,0.039225
4,ENSG00000000460,9.661874,9.255142,0.042097,0.042097


In [84]:
enriched = genes.merge(disease_associations, on='gene_id', how='left')
print(enriched.shape)
enriched.head()

(765313, 7)


,gene_id,control_mean,treatment_mean,suppression,deviation,disease_name,association_score
0,ENSG00000000003,6.880125,8.381852,0.0,0.21827,hypertension,0.431492
1,ENSG00000000003,6.880125,8.381852,0.0,0.21827,low density lipoprotein cholesterol measurement,0.431377
2,ENSG00000000003,6.880125,8.381852,0.0,0.21827,glomerular filtration rate,0.429455
3,ENSG00000000003,6.880125,8.381852,0.0,0.21827,serum creatinine amount,0.345194
4,ENSG00000000003,6.880125,8.381852,0.0,0.21827,body height,0.345194


# Analysis
Get the best number of genes because if we do like 20 then we only get a few gene associations whereas if we do all 78k then there is some random correlation ro whatever kind of like the stock data so we can figure it out using the log2FC

In [99]:
pca_df = pd.read_parquet('pca_df.parquet')
dist_summary = pd.read_csv('dist_summary.csv').set_index('sample_type')

# Identify the Pareto frontier treatments
sorted_df = dist_summary.sort_values('log2FC')
pareto_treatments = []
min_dist = float('inf')
for name, row in sorted_df.iterrows():
    if row['mean_dist'] < min_dist:
        pareto_treatments.append(name)
        min_dist = row['mean_dist']
print("Pareto frontier treatments:", pareto_treatments)

# Load CPM matrix and prepare control baseline
cpm = pd.read_parquet('sample_by_gene.parquet')
gene_cols = [c for c in cpm.columns if c.startswith('ENSG')]
control_mask = cpm['sample_type'] == 'Control'
control_means = cpm.loc[control_mask, gene_cols].mean(axis=0)

# Compute per-gene log2FC per treatment, filter to meaningfully disturbed genes
pseudocount = 0.5
LOG2FC_THRESHOLD = 1.0

off_target_results = {}
for tx in pareto_treatments:
    tx_mask = cpm['sample_type'] == tx
    tx_means = cpm.loc[tx_mask, gene_cols].mean(axis=0)
    log2fc = np.log2((tx_means + pseudocount) / (control_means + pseudocount))
    
    gene_log2fc = pd.DataFrame({
        'gene_id': gene_cols,
        'log2fc': log2fc.values,
        'abs_log2fc': np.abs(log2fc.values),
    })
    
    meaningful = gene_log2fc[gene_log2fc['abs_log2fc'] >= LOG2FC_THRESHOLD]
    off_target_results[tx] = meaningful
    print(f"{tx}: {len(meaningful)} genes with |log2FC| >= {LOG2FC_THRESHOLD}")

# Enrich with trait associations and print summary per treatment
trait_associations = pd.read_parquet('disease_associations.parquet')

for tx, top_genes in off_target_results.items():
    enriched = top_genes.merge(trait_associations, on='gene_id', how='left')
    
    print(f"\n{'='*60}")
    print(f"TREATMENT: {tx}")
    print(f"  log2FC of SNHG14: {dist_summary.loc[tx, 'log2FC']:.3f}")
    print(f"  Distance to Control: {dist_summary.loc[tx, 'mean_dist']:.1f}")
    print(f"  Total meaningfully disturbed genes: {len(top_genes)}")
    print(f"{'='*60}")
    
    top_traits = (enriched
                  .dropna(subset=['disease_name'])
                  .groupby('disease_name')
                  .agg(n_genes=('gene_id', 'nunique'),
                       max_score=('association_score', 'max'))
                  .sort_values(['n_genes', 'max_score'], ascending=False)
                  .head(10))
    print(top_traits)

Pareto frontier treatments: ['nZF139', 'nZF105', 'hATF561', 'hATF567']
nZF139: 64 genes with |log2FC| >= 1.0
nZF105: 61 genes with |log2FC| >= 1.0
hATF561: 25 genes with |log2FC| >= 1.0
hATF567: 62 genes with |log2FC| >= 1.0

TREATMENT: nZF139
  log2FC of SNHG14: -1.088
  Distance to Control: 274.9
  Total meaningfully disturbed genes: 64
                                                    n_genes  max_score
disease_name                                                          
glomerular filtration rate                                5   0.366175
neoplasm                                                  5   0.146619
serum creatinine amount                                   4   0.410488
neurodegenerative disease                                 4   0.383941
body height                                               4   0.344446
high density lipoprotein cholesterol measurement          4   0.332383
aspartate aminotransferase measurement                    3   0.508485
cystatin C measureme

Of the [threshold] top-disturbed genes from this treatment, [n_genes] of them have known associations with [trait]. The strongest single gene-trait link has evidence score [max_score].

In [100]:
import plotly.graph_objects as go

# Collect each frontier treatment's top traits
trait_rows = []
for tx in pareto_treatments:
    top_genes = off_target_results[tx]
    enriched = top_genes.merge(trait_associations, on='gene_id', how='left').dropna(subset=['disease_name'])
    top = (enriched
           .groupby('disease_name')
           .agg(n_genes=('gene_id', 'nunique'), max_score=('association_score', 'max'))
           .sort_values('n_genes', ascending=False)
           .head(15)
           .reset_index())
    top['treatment'] = tx
    trait_rows.append(top)

trait_long = pd.concat(trait_rows, ignore_index=True)

# Normalize by total disturbed genes per treatment
treatment_totals = {tx: len(off_target_results[tx]) for tx in pareto_treatments}
trait_long['pct_of_disturbed'] = trait_long.apply(
    lambda r: r['n_genes'] / treatment_totals[r['treatment']] * 100, axis=1
)

# Pivot both views
count_data = trait_long.pivot_table(
    index='disease_name', columns='treatment', values='n_genes', fill_value=0
)
pct_data = trait_long.pivot_table(
    index='disease_name', columns='treatment', values='pct_of_disturbed', fill_value=0
)

# Sort by total impact (using counts for ordering, applied to both)
order = count_data.sum(axis=1).sort_values(ascending=True).index
count_data = count_data.loc[order]
pct_data = pct_data.loc[order]

# Build figure with both traces, toggle between them via buttons
fig = go.Figure()

# Trace 1: Raw counts (visible by default)
fig.add_trace(go.Heatmap(
    z=count_data.values,
    x=count_data.columns,
    y=count_data.index,
    colorscale='Purples',
    text=count_data.values,
    texttemplate='%{text}',
    textfont=dict(size=11),
    colorbar=dict(title='# disturbed<br>genes linked'),
    visible=True,
    name='Raw counts',
    hovertemplate='Treatment: %{x}<br>Trait: %{y}<br>Genes: %{z}<extra></extra>',
))

# Trace 2: Percentage normalized (hidden by default)
fig.add_trace(go.Heatmap(
    z=pct_data.values,
    x=pct_data.columns,
    y=pct_data.index,
    colorscale='Purples',
    text=pct_data.values,
    texttemplate='%{text:.1f}%',
    textfont=dict(size=11),
    colorbar=dict(title='% of disturbed<br>genes linked'),
    visible=False,
    name='Normalized %',
    hovertemplate='Treatment: %{x}<br>Trait: %{y}<br>Share: %{z:.1f}%<extra></extra>',
))

# Toggle buttons
fig.update_layout(
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.5, xanchor='center',
        y=1.12, yanchor='top',
        showactive=True,
        buttons=[
            dict(label='Raw counts', method='update',
                 args=[{'visible': [True, False]},
                       {'title': 'Trait/Disease Associations — Raw Gene Counts'}]),
            dict(label='Normalized %', method='update',
                 args=[{'visible': [False, True]},
                       {'title': 'Trait/Disease Associations — % of Disturbed Genes per Treatment'}]),
        ],
    )],
    title='Trait/Disease Associations — Raw Gene Counts',
    xaxis_title='Treatment',
    yaxis_title='',
    height=800, width=950,
    template='plotly_white',
)
fig.show()

toggle between views above to see normalized vs the raw counts